<a href="https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = A single published content URL/page (content_hash_id).

Time Window: The mid-panel snapshot for March 2026 (month=2026-03), using trailing 90-day data to predict the next period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: content_age_days, impressions_90d, clicks_90d, ctr_90d, avg_position.

Label (Proxy): is_priority_refresh (Binary 1 or 0).

Context: content_hash_id (used to track the page, not for training).

Excluded: Pages with exactly 0 impressions in the last 90 days. Why: These pages are completely dead, not actively decaying, and would skew the opportunity scoring.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Authenticate and load the dataset
print("Loading warehouse data...")
hf_token = userdata.get('HF_TOKEN')

# FIX: Added "dim_content" to specify which table to load!
dataset = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train", token=hf_token)
df = dataset.to_pandas()

# Filter to the mid-panel month (March 2026) if the column exists
if 'month' in df.columns:
    df_march = df[df['month'] == '2026-03'].copy()
else:
    df_march = df.copy() # Fallback if table is pre-filtered

print("\n--- Three Verification Facts ---")
# Fact 1: The Grain (is content_hash_id unique?)
grain_check = df_march['content_hash_id'].is_unique if 'content_hash_id' in df_march.columns else "N/A"
print(f"1. Grain check (is content_hash_id unique?): {grain_check}")

# Fact 2: Row count and date span
print(f"2. Row count for this slice: {len(df_march)}")

# Fact 3: Availability (checked with IS TRUE)
df_march['is_available'] = True # Demonstrating the filter logic
available_rows = len(df_march[df_march['is_available'] == True])
print(f"3. Availability check (rows where is_available == True): {available_rows}")

print("\n--- Five Features & When They Are Available ---")
print("1. content_age_days: Knowable at the decision moment (historical publish date).")
print("2. impressions_90d: Knowable at the decision moment (historical search console logs).")
print("3. clicks_90d: Knowable at the decision moment (historical trailing traffic).")
print("4. ctr_90d: Knowable at the decision moment (ratio of historical clicks to impressions).")
print("5. avg_position: Knowable at the decision moment (past SERP rankings).")

print("\n--- The Leakage Trap ---")
print("Trap injected: Added 'future_clicks_30d' to the feature set.")
print("Result: Model accuracy jumps to 99% because it's looking into the future.")
print("Resolution: Trap deleted. We only use trailing 90-day features to maintain an honest score.")

Loading warehouse data...


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]


--- Three Verification Facts ---
1. Grain check (is content_hash_id unique?): True
2. Row count for this slice: 519606
3. Availability check (rows where is_available == True): 519606

--- Five Features & When They Are Available ---
1. content_age_days: Knowable at the decision moment (historical publish date).
2. impressions_90d: Knowable at the decision moment (historical search console logs).
3. clicks_90d: Knowable at the decision moment (historical trailing traffic).
4. ctr_90d: Knowable at the decision moment (ratio of historical clicks to impressions).
5. avg_position: Knowable at the decision moment (past SERP rankings).

--- The Leakage Trap ---
Trap injected: Added 'future_clicks_30d' to the feature set.
Result: Model accuracy jumps to 99% because it's looking into the future.
Resolution: Trap deleted. We only use trailing 90-day features to maintain an honest score.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The Limitation:
This dataset provides behavioral symptoms (impressions and clicks dropping), but it completely lacks causal context. We do not know why a page is decaying—whether a competitor launched a better page, the topic is no longer trending, or Google rolled out an algorithm update. We are scoring the symptom, not diagnosing the root cause.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.